# Signpost-D model compatibility check: can a candidate base run our recipe on a T4?

A short check of each candidate before any screen or training run, on **one T4**, no training run and no benchmark. For each model in `MODELS` (the current base first, as the reference):

1. **Load** in fp16: which loader works (text-only causal LM, or the image-text class with its language model inside), load time, GPU memory, and for Qwen3.5 whether the fast linear-attention kernels are importable.
2. **Prompt:** notebook 11's packed prompt through the model's own chat template with thinking off; the end of the prompt is printed, and a prompt that ends inside an open thinking block is flagged. The answer letters ` A` … ` Z` must each be one token.
3. **fp16 health:** the largest activation in every layer on the longest case (fp16 overflows at 65,504), and whether any output is NaN or infinite. All three candidates were released in bf16, which a T4 cannot run.
4. **A few cases, untrained:** `N_CASES` shift_dev cases, all their questions in one pass, as notebook 11 scores them: accuracy (a sanity check, not a decision; too few cases) and the mean top probability.
5. **Timing against the budget:** each of those cases at batch 1 after a warm-up: milliseconds per case and per question (median, p95) against **100 ms per question**.
6. **LoRA:** notebook 11's target modules attached (count by module name), then one training step on the longest case with gradient checkpointing, fp32 LoRA weights and fp16 autocast: loss and gradient norm finite, peak memory.

Each stage is reported on its own, so one failure does not hide the rest; the table at the end compares the models. Nothing is saved except the summary JSON; nothing is uploaded.

In [1]:
# Parameters

MODELS = [                               # the current base first, as the reference
    "Qwen/Qwen3-4B-Instruct-2507",
    "Qwen/Qwen3.5-4B",
    "openbmb/MiniCPM5-2B",
    "ibm-granite/granite-4.2-3b",
]
N_CASES = 30                             # shift_dev cases for the accuracy sanity check and the timing (a fixed sample)
WARMUP = 3                               # untimed passes before timing
BUDGET_MS = 100                          # per question, T4, batch 1
INSTALL_FLA = True                       # install flash-linear-attention (Triton) for Qwen3.5's linear-attention layers
RUN_NAME = "13_model_check"

In [2]:
# Cell 1 — Setup: packages, data, the prompt (as notebook 11)

!pip uninstall -y -q torchao
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
if INSTALL_FLA:
    !pip install -q --no-deps flash-linear-attention

import gc, json, math, os, random, statistics, subprocess, time, traceback
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import torch
import transformers
from huggingface_hub import HfApi

HF_TOKEN, HF_TOKEN_FROM = os.environ.get("HF_TOKEN") or None, "the environment"     # never printed or written to the notebook
if HF_TOKEN is None:
    HF_TOKEN_FROM = "NOT FOUND"
    try:
        from kaggle_secrets import UserSecretsClient
        for label in ("HF_TOKEN", "hf_token"):
            try:
                HF_TOKEN = UserSecretsClient().get_secret(label) or None
            except Exception as e:                    # the secret is not attached to this notebook, or no Kaggle session
                HF_TOKEN_FROM = f"NOT FOUND (Kaggle secret {label!r}: {type(e).__name__}: {str(e)[:80]})"
            if HF_TOKEN:
                HF_TOKEN_FROM = f"Kaggle secret {label!r}"
                break
    except ImportError:
        HF_TOKEN_FROM = "NOT FOUND (no kaggle_secrets module: not a Kaggle session)"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face token: {HF_TOKEN_FROM}")   # where it came from; the token itself is never printed
DEVICE = torch.device("cuda:0")
LETTERS = [chr(65 + i) for i in range(26)]
LORA_TARGETS = r"^(?!.*(visual|vision)).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj)$"
SYSTEM_PACKED = ("You read a case and answer every decision question about it. For each question, write its number and the "
                 "letter of the option you choose.")
api = HfApi()
print(f"torch {torch.__version__} | transformers {transformers.__version__} | {torch.cuda.get_device_name(0)}")

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data")
if not (DATA_DIR / "signpost_d").exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data"


def render(q):
    cr = q.get("criteria") or {}
    if q["type"] == "noul":
        keys = ["true", "false"]
        opts = {k: cr.get(k) or "" for k in keys}
    elif q["type"] == "choice":
        keys = list(cr)
        opts = {k.replace("_", " "): v or "" for k, v in cr.items()}
    else:
        keys = [str(i) for i in range(len(cr))]
        opts = {str(c): "" for c in cr}
    return opts, keys


def case_items(c):
    state = c["state"] if isinstance(c["state"], str) else json.dumps(c["state"], ensure_ascii=False)
    out = []
    for qid, q in c["questions"].items():
        opts, keys = render(q)
        g = c["gold"][qid]["probabilities"]
        out.append({"task": "d_" + q["type"], "state": state, "question": q["instructions"], "options": opts,
                    "soft": {n: g[k] for n, k in zip(opts, keys)}})
    return out


def question_block(item):
    names, task = list(item["options"]), item["task"]
    if task == "d_noul":
        head = f"Statement: {item['question']}\nIs the statement true or false?"
    elif task == "d_score":
        head = f"Question: {item['question']}\nLevels, lowest first:"
    else:
        head = f"Question: {item['question']}\nOptions:"
    n, lines = len(names), []
    for i, nm in enumerate(names):
        pos = ""
        if task == "d_score":
            pos = f"level {i + 1} of {n}" + (" (lowest)" if i == 0 else " (highest)" if i == n - 1 else "") + ": "
        desc = item["options"][nm]
        lines.append(f"{LETTERS[i]}. {pos}{nm}" + (f": {desc}" if desc else ""))
    return head + "\n" + "\n".join(lines)


shift = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / "independent_v1" / "shift_dev.jsonl")]
CASES = [case_items(c) for c in random.Random(0).sample(shift, N_CASES)]
print(f"{len(CASES)} shift_dev cases, {sum(len(c) for c in CASES)} questions "
      f"({statistics.mean(len(c) for c in CASES):.1f} per case)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 33.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 32.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.5/403.5 kB 8.4 MB/s eta 0:00:00:00:01
torch 2.11.0+cu128 | transformers 5.17.0 | Tesla T4


From https://github.com/skamalj/llms-the-hard-way
 * branch            signpost-typed-decisions -> FETCH_HEAD
 + 3cd20e4...1059819 signpost-typed-decisions -> origin/signpost-typed-decisions  (forced update)


Your branch is up to date with 'origin/signpost-typed-decisions'.
30 shift_dev cases, 136 questions (4.5 per case)


Reset branch 'signpost-typed-decisions'


In [3]:
# Cell 2 — The checks for one model

from transformers import AutoModelForCausalLM, AutoTokenizer


def gpu_gb():
    return round(torch.cuda.max_memory_allocated(DEVICE) / 1e9, 2)


class Probe:
    def __init__(self, name):
        self.name = name
        self.rev = api.model_info(name, token=HF_TOKEN).sha
        self.tok = AutoTokenizer.from_pretrained(name, revision=self.rev, token=HF_TOKEN)
        kw = dict(revision=self.rev, token=HF_TOKEN, dtype=torch.float16, device_map={"": 0})
        try:
            self.m, self.loader = AutoModelForCausalLM.from_pretrained(name, **kw), "AutoModelForCausalLM"
        except (ValueError, KeyError) as e:                    # an image-text model with no causal-LM mapping
            from transformers import AutoModelForImageTextToText
            print(f"  AutoModelForCausalLM refused ({str(e)[:120]}); trying AutoModelForImageTextToText")
            self.m, self.loader = AutoModelForImageTextToText.from_pretrained(name, **kw), "AutoModelForImageTextToText"
        self.m.eval()
        self.m.config.use_cache = False
        base = self.m.model
        self.inner = base.language_model if hasattr(base, "language_model") else base
        self.head_w = self.m.get_output_embeddings().weight
        self.layers = self.inner.layers

    def prefix(self, items):
        user = "Case:\n" + items[0]["state"] + "\n\n" + "\n\n".join(f"[{i + 1}] " + question_block(it) for i, it in enumerate(items))
        msgs = [{"role": "system", "content": SYSTEM_PACKED}, {"role": "user", "content": user}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    def case_ids(self, items):
        ids, pos = self.tok(self.prefix(items), add_special_tokens=False)["input_ids"], []
        for i in range(len(items)):
            ids = ids + self.tok(f"{i + 1}:", add_special_tokens=False)["input_ids"]
            pos.append(len(ids) - 1)
            ids = ids + self.tok(" _\n", add_special_tokens=False)["input_ids"]
        return ids, pos

    def letter_logits(self, items, grad=False):
        ids, pos = self.case_ids(items)
        x = torch.tensor([ids], device=DEVICE)
        with torch.set_grad_enabled(grad), torch.autocast(device_type="cuda", dtype=torch.float16):
            h = self.inner(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False).last_hidden_state
        return h[0, pos].float() @ self.head_w[self.label_ids].float().T, len(ids)


def check(name):
    r, t0 = {"model": name}, time.perf_counter()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    try:
        p = Probe(name)
    except Exception as e:
        r["load"] = f"FAILED: {type(e).__name__}: {str(e)[:300]}"
        print(traceback.format_exc()[-1500:])
        return r, None
    r.update(load="ok", loader=p.loader, revision=p.rev[:8], load_s=round(time.perf_counter() - t0), weights_gb=gpu_gb(),
             params_b=round(sum(x.numel() for x in p.m.parameters()) / 1e9, 2),
             text_params_b=round(sum(x.numel() for x in p.inner.parameters()) / 1e9, 2), layers=len(p.layers))
    if "Qwen3.5" in name:
        for mod in ("fla", "causal_conv1d"):
            try:
                __import__(mod)
                r[f"kernel_{mod}"] = "importable"
            except Exception as e:
                r[f"kernel_{mod}"] = f"missing ({type(e).__name__})"
    print(f"  loaded via {p.loader} in {r['load_s']} s | {r['params_b']} B params ({r['text_params_b']} B text) | "
          f"{r['layers']} layers | {r['weights_gb']} GB")

    # 2. prompt and letters
    tail = p.prefix(CASES[0])[-160:]
    print(f"  prompt ends with: {tail!r}")
    open_think = tail.rstrip().endswith("<think>") or (tail.count("<think>") > tail.count("</think>"))
    r["prompt"] = "ENDS INSIDE AN OPEN THINKING BLOCK" if open_think else "ok"
    ids = [p.tok(" " + L, add_special_tokens=False)["input_ids"] for L in LETTERS]
    r["letters_one_token"] = all(len(x) == 1 for x in ids)
    if not r["letters_one_token"]:
        print(f"  letters are not single tokens: {ids[:5]}")
        return r, p
    p.label_ids = torch.tensor([x[0] for x in ids], device=DEVICE)

    # 3. fp16 health on the longest case
    longest = max(CASES, key=lambda c: len(p.case_ids(c)[0]))
    peaks, hooks = [], []
    for layer in p.layers:
        hooks.append(layer.register_forward_hook(
            lambda mod, inp, out: peaks.append(float((out[0] if isinstance(out, tuple) else out).detach().abs().float().max()))))
    try:
        with torch.no_grad():
            lg, n_tok = p.letter_logits(longest)
        r.update(longest_case_tokens=n_tok, max_activation=round(max(peaks)), worst_layer=int(max(range(len(peaks)), key=peaks.__getitem__)),
                 nonfinite=bool(not torch.isfinite(lg).all()) or not all(math.isfinite(v) for v in peaks))
    finally:
        for h in hooks:
            h.remove()
    r["fp16"] = "NaN/inf" if r["nonfinite"] else ("near overflow" if r["max_activation"] > 30000 else "ok")
    print(f"  fp16: largest activation {r['max_activation']} (layer {r['worst_layer']}, limit 65,504) on {n_tok} tokens | {r['fp16']}")

    # 4 and 5. a few cases, untrained, each timed at batch 1
    with torch.no_grad():
        for c in CASES[:WARMUP]:
            p.letter_logits(c)
        ok, tops, case_ms, q_ms = [], [], [], []
        for c in CASES:
            torch.cuda.synchronize(DEVICE)
            t1 = time.perf_counter()
            lg, _ = p.letter_logits(c)
            torch.cuda.synchronize(DEVICE)
            ms = (time.perf_counter() - t1) * 1000
            case_ms.append(ms)
            q_ms.append(ms / len(c))
            for i, it in enumerate(c):
                pr = torch.softmax(lg[i, : len(it["options"])], 0).tolist()
                names = list(it["options"])
                ok.append(names[max(range(len(pr)), key=pr.__getitem__)] == max(it["soft"], key=it["soft"].get))
                tops.append(max(pr))
    pct = lambda xs, q: round(sorted(xs)[min(len(xs) - 1, int(q * len(xs)))], 1)
    r.update(accuracy_untrained=round(statistics.mean(ok), 3), mean_top_p=round(statistics.mean(tops), 3), questions=len(ok),
             ms_case_p50=pct(case_ms, 0.5), ms_case_p95=pct(case_ms, 0.95), ms_question_p50=pct(q_ms, 0.5),
             ms_question_p95=pct(q_ms, 0.95))
    r["within_budget"] = r["ms_question_p95"] <= BUDGET_MS
    print(f"  untrained: accuracy {r['accuracy_untrained']} on {len(ok)} questions, mean top {r['mean_top_p']} | "
          f"per case p50 {r['ms_case_p50']} ms p95 {r['ms_case_p95']} | per question p50 {r['ms_question_p50']} p95 "
          f"{r['ms_question_p95']} ms (budget {BUDGET_MS}) -> {'within' if r['within_budget'] else 'OVER'}")

    # 6. LoRA: targets and one training step on the longest case
    from peft import LoraConfig, get_peft_model
    try:
        p.m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        p.m = get_peft_model(p.m, LoraConfig(r=16, lora_alpha=16, lora_dropout=0.05, target_modules=LORA_TARGETS))
        counts = {}
        for n, mod in p.m.named_modules():
            if hasattr(mod, "lora_A") and isinstance(getattr(mod, "lora_A"), torch.nn.ModuleDict) and len(mod.lora_A):
                k = n.split(".")[-1]
                counts[k] = counts.get(k, 0) + 1
        params = [x for x in p.m.parameters() if x.requires_grad]
        for x in params:
            x.data = x.data.float()
        base = p.m.get_base_model().model
        p.inner = base.language_model if hasattr(base, "language_model") else base
        p.m.train()
        torch.cuda.reset_peak_memory_stats(DEVICE)
        scaler = torch.amp.GradScaler("cuda")
        opt = torch.optim.AdamW(params, lr=1e-4)
        t1 = time.perf_counter()
        lg, _ = p.letter_logits(longest, grad=True)
        loss = 0.0
        for i, it in enumerate(longest):
            target = torch.tensor([it["soft"][n] for n in it["options"]], device=DEVICE)
            loss = loss - (target / target.sum() * torch.log_softmax(lg[i, : len(it["options"])], 0)).sum()
        loss = loss / len(longest)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        gn = float(torch.nn.utils.clip_grad_norm_(params, 1.0))
        scaler.step(opt); scaler.update()
        torch.cuda.synchronize(DEVICE)
        r.update(lora="ok", lora_modules=counts, lora_trainable_m=round(sum(x.numel() for x in params) / 1e6, 1),
                 train_loss=round(float(loss), 4), grad_norm=round(gn, 3), train_step_s=round(time.perf_counter() - t1, 2),
                 train_peak_gb=gpu_gb(), train_finite=math.isfinite(float(loss)) and math.isfinite(gn))
        print(f"  LoRA: {r['lora_trainable_m']} M trainable on {counts} | step loss {r['train_loss']} grad norm {r['grad_norm']} | "
              f"{r['train_step_s']} s | peak {r['train_peak_gb']} GB | finite {r['train_finite']}")
    except Exception as e:
        r["lora"] = f"FAILED: {type(e).__name__}: {str(e)[:300]}"
        print(traceback.format_exc()[-1500:])
    return r, p

In [4]:
# Cell 3 — Every model in turn, then the table

RESULTS = []
for name in MODELS:
    print(f"===== {name} =====")
    try:
        r, p = check(name)
    except Exception as e:
        r, p = {"model": name, "error": f"{type(e).__name__}: {str(e)[:300]}"}, None
        print(traceback.format_exc()[-1500:])
    RESULTS.append(r)
    p = None
    gc.collect()
    torch.cuda.empty_cache()

cols = ["load", "loader", "params_b", "layers", "prompt", "letters_one_token", "fp16", "max_activation", "accuracy_untrained",
        "mean_top_p", "ms_question_p50", "ms_question_p95", "within_budget", "lora", "train_finite", "train_peak_gb"]
print("\n" + " | ".join(["model"] + cols))
for r in RESULTS:
    print(" | ".join([r["model"].split("/")[-1]] + [str(r.get(c, "-")) for c in cols]))
for r in RESULTS:
    extra = {k: v for k, v in r.items() if k.startswith("kernel_") or k == "lora_modules" or k == "error"}
    if extra:
        print(f"{r['model']}: {extra}")
json.dump(RESULTS, open(f"{RUN_NAME}_summary.json", "w"), indent=1)
print("CHECK_JSON " + json.dumps(RESULTS, separators=(",", ":")))

===== Qwen/Qwen3-4B-Instruct-2507 =====


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

  loaded via AutoModelForCausalLM in 44 s | 3.66 B params (3.4 B text) | 40 layers | 7.34 GB
  prompt ends with: 'C. level 3 of 4: Mostly ready: only minor gaps remain\nD. level 4 of 4 (highest): Ready: nothing material remains<|im_end|>\n<|im_start|>assistant\n<think></think>'
  fp16: largest activation 12384 (layer 6, limit 65,504) on 1439 tokens | ok
  untrained: accuracy 0.544 on 136 questions, mean top 0.738 | per case p50 1216.4 ms p95 1723.1 | per question p50 262.5 p95 409.9 ms (budget 100) -> OVER
  LoRA: 31.1 M trainable on {'q_proj': 40, 'k_proj': 40, 'v_proj': 40, 'o_proj': 40, 'gate_proj': 40, 'up_proj': 40, 'down_proj': 40} | step loss 1.1304 grad norm 34.232 | 5.64 s | peak 9.19 GB | finite True

model | load | loader | params_b | layers | prompt | letters_one_token | fp16 | max_activation | accuracy_untrained | mean_top_p | ms_question_p50 | ms_question_p95 | within_budget | lora | train_finite | train_peak_gb
Qwen3-4B-Instruct-2507 | ok | AutoModelForCausalLM | 4.02 | 